# 05 Productivity Prediction

Detects a productivity-like numeric target, trains three regressors, reports MAE/RMSE/R2, and saves the best model to `ai/models`.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import joblib
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def detect_productivity_target(df):
    candidates = []
    for col in df.columns:
        low = col.lower().replace('_', ' ')
        if pd.api.types.is_numeric_dtype(df[col]) and any(k in low for k in ['productivity score', 'productivity_score', 'productivity', 'performance score', 'efficiency']):
            candidates.append(col)
    return candidates[0] if candidates else None

def build_preprocessor(X):
    numeric = X.select_dtypes(include='number').columns.tolist()
    categorical = X.select_dtypes(exclude='number').columns.tolist()
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])

def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {'MAE': mean_absolute_error(y_test, pred), 'RMSE': float(np.sqrt(mean_squared_error(y_test, pred))), 'R2': r2_score(y_test, pred)}


In [ ]:
trained_any = False
for path in discover_data_files():
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path).dropna(axis=1, how='all')
        target = detect_productivity_target(df)
        if not target:
            print('No suitable productivity target found.')
            continue
        data = df.dropna(subset=[target]).copy()
        if len(data) < 20:
            print(f'Not enough rows after target filtering: {len(data)}')
            continue
        X, y = data.drop(columns=[target]), data[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        models = {'Linear Regression': LinearRegression(), 'Random Forest': RandomForestRegressor(n_estimators=150, random_state=42), 'Decision Tree': DecisionTreeRegressor(random_state=42)}
        rows, fitted = [], {}
        for name, estimator in models.items():
            pipe = Pipeline([('preprocess', build_preprocessor(X)), ('model', estimator)])
            metrics = evaluate(pipe, X_train, X_test, y_train, y_test)
            rows.append({'model': name, **metrics})
            fitted[name] = pipe
        results = pd.DataFrame(rows).sort_values(['RMSE', 'MAE'], ascending=True)
        display(results)
        best_name = results.iloc[0]['model']
        output = MODEL_DIR / f'{safe_name(path).rsplit(".", 1)[0]}_{best_name.lower().replace(" ", "_")}.joblib'
        joblib.dump(fitted[best_name], output)
        print('Target:', target)
        print('Best model:', best_name)
        print('Saved:', output.relative_to(PROJECT_ROOT))
        trained_any = True
    except Exception as exc:
        print(f'Model training failed for {path.name}: {exc}')
if not trained_any:
    print('No productivity prediction model was trained because no dataset contained a suitable numeric productivity target.')
